# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mubashir-dev751/starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [10]:
!pip install duckdb -q

import duckdb
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from google.colab import userdata

hf_token = userdata.get('hf_token').strip()
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

Unit of analysis (Grain): Exactly one content item (content_hash_id) per client (client_hash_id) on a single calendar day (report_date).
Time window: The full month of March 2026 (2026-03-01 to 2026-03-31).
Objective: Predict next-day organic search engagement (has_clicks_next_day).

In [11]:
query_init = """
SELECT
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(*) AS total_rows
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet';
"""
df_init = con.execute(query_init).df()
print("Initial time window check:")
display(df_init)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Initial time window check:


,min_date,max_date,total_rows
0,2026-03-01,2026-03-31,9841378


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [12]:
query_fields = """
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_avg_position,
    gsc_clicks,
    ga4_pageviews,
    ga4_engaged_sessions,
    ga4_data_available
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
LIMIT 5;
"""
df_fields = con.execute(query_fields).df()
print("Field classification sample:")
display(df_fields)

Field classification sample:


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,gsc_clicks,ga4_pageviews,ga4_engaged_sessions,ga4_data_available
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,3.350000,0,<NA>,<NA>,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0.000000,0,<NA>,<NA>,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,4.928000,1,<NA>,<NA>,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,4.000000,0,<NA>,<NA>,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,2.272727,0,<NA>,<NA>,<NA>


Label: has_clicks_next_day (derived from gsc_clicks on day t+1).
Features: gsc_impressions, gsc_avg_position, ga4_pageviews, ga4_engaged_sessions, and a missing position flag. (All knowable on day t).
Context: report_date, client_hash_id, content_hash_id.
Excluded: trend_direction and trend_pct (causes future leakage).

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [13]:
query_grain = """
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
GROUP BY report_date, client_hash_id, content_hash_id
HAVING c > 1 LIMIT 5;
"""
df_grain = con.execute(query_grain).df()
assert len(df_grain) == 0, "Grain violation detected!"
print("Fact 1: Grain holds (0 duplicates).")

query_span = """
SELECT COUNT(*) AS total_rows, MIN(report_date) AS start_date, MAX(report_date) AS end_date
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet';
"""
print("\nFact 2: Span")
display(con.execute(query_span).df())

query_avail = """
SELECT COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS available_rows
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet';
"""
print("\nFact 3: Availability")
display(con.execute(query_avail).df())

query_features = """
WITH base AS (
    SELECT
        client_hash_id, content_hash_id, report_date,
        gsc_impressions, gsc_avg_position, ga4_pageviews, ga4_engaged_sessions,
        LEAD(gsc_clicks, 1) OVER (PARTITION BY client_hash_id, content_hash_id ORDER BY report_date) AS clicks_tomorrow
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    WHERE ga4_data_available IS TRUE
)
SELECT
    gsc_impressions AS feat_impressions,
    CASE WHEN gsc_avg_position = 0 THEN 100 ELSE gsc_avg_position END AS feat_avg_pos,
    CASE WHEN gsc_avg_position = 0 THEN 1 ELSE 0 END AS feat_no_pos_flag,
    ga4_pageviews AS feat_pageviews,
    ga4_engaged_sessions AS feat_engaged_sessions,
    clicks_tomorrow AS LEAKED_future_clicks,
    CASE WHEN clicks_tomorrow > 0 THEN 1 ELSE 0 END AS label_clicks_next_day
FROM base WHERE clicks_tomorrow IS NOT NULL;
"""
df = con.execute(query_features).df()

features_leak = ['feat_impressions', 'feat_avg_pos', 'feat_no_pos_flag', 'feat_pageviews', 'feat_engaged_sessions', 'LEAKED_future_clicks']
X_leak = df[features_leak]
y = df['label_clicks_next_day']
X_train, X_test, y_train, y_test = train_test_split(X_leak, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=20, max_depth=5, random_state=42)
model.fit(X_train, y_train)
print(f"\nROC-AUC WITH Leakage: {roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]):.4f}")

features_honest = ['feat_impressions', 'feat_avg_pos', 'feat_no_pos_flag', 'feat_pageviews', 'feat_engaged_sessions']
X_honest = df[features_honest]
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(X_honest, y, test_size=0.2, random_state=42)
model.fit(X_train_h, y_train_h)
print(f"ROC-AUC Honest Baseline: {roc_auc_score(y_test_h, model.predict_proba(X_test_h)[:, 1]):.4f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Fact 1: Grain holds (0 duplicates).

Fact 2: Span


,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31



Fact 3: Availability


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,413966


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


ROC-AUC WITH Leakage: 1.0000
ROC-AUC Honest Baseline: 0.7905


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [14]:
query_limit_check = """
SELECT
    ga4_data_available,
    COUNT(*) AS row_count,
    AVG(ga4_pageviews) AS avg_pageviews,
    AVG(ga4_engaged_sessions) AS avg_engaged_sessions
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
GROUP BY ga4_data_available;
"""
df_limit_check = con.execute(query_limit_check).df()
print("Verification of GA4 Availability Limits:")
display(df_limit_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Verification of GA4 Availability Limits:


,ga4_data_available,row_count,avg_pageviews,avg_engaged_sessions
0,<NA>,3018741,NaN,NaN
1,False,6408671,0.000000,0.000000
2,True,413966,3.586903,0.071385


Data Limits: Rows before a client's analytics tracking started are zero-filled. Filtering by ga4_data_available is mandatory to avoid training on false zeroes. gsc_avg_position = 0 implies missing data, not a top rank.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.